In [1]:
from notebooks.consts import PROCESSED_CSV
from notebooks.features.feature_extraction import load_all_features
from notebooks.preprocessing import preprocess_aso_data

aso_df = load_all_features(version='v2')

In [2]:
aso_data = preprocess_aso_data(PROCESSED_CSV, include_smiles=False)

Adding cell line depmap name proxy
Adding a depmap column
Preprocessing complete. Final valid rows: 18153


In [3]:
import pandas as pd

merged_df = pd.merge(aso_df, aso_data, on='index_v2')

In [4]:
from tauso.data.consts import INHIBITION, CELL_LINE

features_to_ignore = ['log_inhibition', INHIBITION, 'ISIS', 'index_v2', 'replicate_count', 'total_replicate_count', 'Unnamed: 0', 'probe_count']

In [5]:
from tauso.data.consts import MODIFICATION


def add_chemistry_features(df, mod_col_name):
    """
    Adds binary flags (0/1) for 'cet', 'moe', and 'lna' based on the modification string.
    """
    df = df.copy()

    # 1. Initialize columns to 0
    df['cet'] = 0
    df['moe'] = 0
    df['lna'] = 0

    # 2. Set to 1 where the substring exists
    # We use case=False just to be safe, though your data seems consistent.

    # Matches 'cEt/...' and '(S)-cEt/...'
    df.loc[df[mod_col_name].str.contains('cEt', case=False, na=False), 'cet'] = 1

    # Matches 'MOE/...'
    df.loc[df[mod_col_name].str.contains('MOE', case=False, na=False), 'moe'] = 1

    # Matches 'LNA/...'
    df.loc[df[mod_col_name].str.contains('LNA', case=False, na=False), 'lna'] = 1

    return df

# --- Usage Example ---
# Assuming 'filtered_data' is the dataframe from your snippet
# and MODIFICATION is the variable holding the column name string (e.g. 'Modification')

merged_df = add_chemistry_features(merged_df, MODIFICATION)

# Quick check
print(merged_df[[MODIFICATION, 'cet', 'moe', 'lna']].head())

                  Modification  cet  moe  lna
0  cEt/5-methylcytosines/deoxy    1    0    0
1  MOE/5-methylcytosines/deoxy    0    1    0
2  MOE/5-methylcytosines/deoxy    0    1    0
3  MOE/5-methylcytosines/deoxy    0    1    0
4  MOE/5-methylcytosines/deoxy    0    1    0


In [6]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.feature_selection import mutual_info_regression
from tauso.data.consts import CELL_LINE, CANONICAL_GENE, INHIBITION
import warnings

# ---------------------------------------------------------
# 1. PREP DATA & DERIVE EC50
# ---------------------------------------------------------
warnings.filterwarnings('ignore')
df = merged_df.copy()
VOLUME_COL = 'ASO_volume(nM)'
TARGET_COL = INHIBITION

# Clip and Derive Log_EC50 (Targeting n=1 for the baseline)
# Formula: EC50 = Vol * (1/Inh - 1)
inh_clipped = df[TARGET_COL].clip(1, 99) / 100.0
df['log_ec50'] = np.log10(df[VOLUME_COL] * ( (1.0 / inh_clipped) - 1.0 ))

# Create Cohorts and Filter for Robustness
df['cohort'] = df[CELL_LINE].astype(str) + '_' + df[CANONICAL_GENE].astype(str)
counts = df['cohort'].value_counts()
df_robust = df[df['cohort'].isin(counts[counts >= 40].index)].copy()

# ---------------------------------------------------------
# 2. DEFINE FEATURES
# ---------------------------------------------------------
# Exclude metadata and non-features
ignore = [TARGET_COL, VOLUME_COL, 'log_ec50', 'cohort', CELL_LINE, CANONICAL_GENE,
          'index_v2', 'Unnamed: 0', 'ISIS', 'replicate_count', 'total_replicate_count']

# Focus on Intrinsic Features (Exclude OT_ as discussed, or keep for full discovery)
candidate_features = [c for c in df_robust.select_dtypes(include=[np.number]).columns
                      if c not in ignore and not c.startswith('OT_')]

print(f"Running discovery on {len(candidate_features)} features across {df_robust['cohort'].nunique()} cohorts...")

# ---------------------------------------------------------
# 3. CALCULATE METRICS
# ---------------------------------------------------------
discovery_results = []
X = df_robust[candidate_features].fillna(0) # Simple fill for MI
y = df_robust['log_ec50']

# Mutual Information (Captures complex non-linear physics)
print("Calculating Mutual Information (this may take a minute)...")
mi_scores = mutual_info_regression(X, y)
mi_dict = dict(zip(candidate_features, mi_scores))

for feat in candidate_features:
    # A. Global Spearman
    glob_s, _ = spearmanr(df_robust[feat], y)

    # B. Consistency (Average Intra-Cohort Spearman)
    intra_corrs = []
    for name, group in df_robust.groupby('cohort'):
        if len(group) > 5:
            s, _ = spearmanr(group[feat], group['log_ec50'])
            if not np.isnan(s):
                intra_corrs.append(s)
    consistency = np.mean(intra_corrs) if intra_corrs else 0

    discovery_results.append({
        'Feature': feat,
        'Global_Spearman': glob_s,
        'Mutual_Info': mi_dict[feat],
        'Intra_Consistency': consistency,
        'Abs_Global': abs(glob_s)
    })


Running discovery on 420 features across 14 cohorts...
Calculating Mutual Information (this may take a minute)...


In [7]:
# ---------------------------------------------------------
# 4. RANKING TABLE
# ---------------------------------------------------------
results_table = pd.DataFrame(discovery_results).sort_values('Intra_Consistency', ascending=False)
print("\nTOP 25 BIOLOGICAL FEATURES (Ranked by Intra-Cohort Consistency):")
print(results_table[['Feature', 'Intra_Consistency', 'Global_Spearman', 'Mutual_Info']].head(50))


TOP 25 BIOLOGICAL FEATURES (Ranked by Intra-Cohort Consistency):
                                          Feature  Intra_Consistency  \
16                               LNA_DIFF_37_HYBR           0.219715   
28                           Modification_in_core           0.180386   
244                              Sequence_at_skew           0.170875   
41                  PSDNA_RNA_MD_37_GB_TOTAL_HYBR           0.159549   
42                  PSDNA_RNA_MD_37_PB_TOTAL_HYBR           0.157331   
19                         MOE_DIFF_37_MD_PB_HYBR           0.149758   
238  RNaseH1_score_dinucleotide_R4a_dinuc_dynamic           0.139519   
18                         MOE_DIFF_37_MD_GB_HYBR           0.135414   
416                                sense_length_y           0.134525   
399                                sense_length_x           0.134525   
235                     RNaseH1_score_R4a_dynamic           0.128297   
33            Modification_min_distance_to_3prime           0.122520  

In [19]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import GroupKFold
from scipy.stats import spearmanr
from collections import Counter
import warnings

# 1. STRICT IMPORTS
from tauso.data.consts import INHIBITION, CELL_LINE, CANONICAL_GENE

# 2. CONFIGURATION
warnings.filterwarnings('ignore')
VOLUME_COL = 'ASO_volume(nM)'
MIN_SAMPLES = 40
MAX_STEPS = 10
CORR_LIMIT = 0.7
N_ROUNDS = 5

# Liver Definition
LIVER_CELL_LINES = ['HepG2', 'HepaRG', 'SNU-449']

# Full Candidate Pool
biological_features = [
    "LNA_DIFF_37_HYBR", "Modification_in_core", "Sequence_at_skew",
    "PSDNA_RNA_MD_37_GB_TOTAL_HYBR", "PSDNA_RNA_MD_37_PB_TOTAL_HYBR",
    "MOE_DIFF_37_MD_PB_HYBR", "RNaseH1_score_dinucleotide_R4a_dinuc_dynamic",
    "MOE_DIFF_37_MD_GB_HYBR", "sense_length_x",
    "RNaseH1_score_R4a_dynamic", "Modification_min_distance_to_3prime",
    "CET_DIFF_37_HYBR", "RNaseH1_score_R4b_dynamic",
    "access_120flank_20access_13seed_sizes",
    "RBP_PIWIL1_expr_aff_50", "Modification_evenness",
    "RNaseH1_score_dinucleotide_R7_dinuc_dynamic", "off_target_single_ACTB_c1200",
    "off_target_single_ACTB_c1000", "RBP_ESRP2_expr_aff_50",
    "off_target_score_general_MECH_n25_c1200", "RBP_SRSF5_expr_aff_50",
    "Sequence_hairpin_score", "RBP_CELF1_expr_aff_50",
    "Sequence_dispersed_repeats_score",
    "Sequence_purine_content", "off_target_single_RPL7_c1000",
    "Sequence_hairpin_tm", "RBP_TIA1_expr_aff_50",
    "Sequence_tandem_repeats_score", "Sequence_homooligo_count",
    "off_target_single_PPIA_c1000", "off_target_score_specific_MECH_n25_c1200",
    "RBP_DHX58_expr_aff_50", "off_target_single_PPIA_c1200",
    "RNaseH1_score_R7_dynamic", "off_target_score_general_MECH_n25_c1000",
    "off_target_score_specific_MECH_n25_c1100", "off_target_single_RNASEH1_c1200",
    "off_target_single_RPL7_c800", "RBP_RBM8A_expr_aff_50",
    "RBP_GRSF1_expr_aff_50", "Modification_mean_gap",
    "Sequence_6_palindromic", "RBP_RC3H1_expr_aff_50",
    "RBP_PPRC1_expr_aff_50", "off_target_single_PPIA_c800",
    'moe', 'cet', 'lna'
]

# 3. ROBUST HELPER FUNCTIONS
def safe_spearman(y_true, y_pred):
    """Returns 0.0 if predictions are constant (std=0), avoiding NaN."""
    if np.std(y_pred) < 1e-9 or np.std(y_true) < 1e-9:
        return 0.0
    return spearmanr(y_true, y_pred)[0]

def michaelis_menten(vol, ec50):
    return vol / (vol + ec50)

def fit_baseline_ec50(vol, inh):
    # Fit median EC50
    inh_clipped = np.clip(inh, 0.01, 0.99)
    estimated_ec50s = vol * ((1.0 / inh_clipped) - 1.0)
    median_ec50 = np.nanmedian(estimated_ec50s)

    # Safety: If median is NaN (e.g. constant data), return 1.0
    if np.isnan(median_ec50) or median_ec50 <= 0:
        return 1.0
    return median_ec50

def get_residuals(df, km=None):
    vol = df[VOLUME_COL].values
    inh = df[INHIBITION].values / 100.0
    if km is None: km = fit_baseline_ec50(vol, inh)
    base_pred = michaelis_menten(vol, km)
    return inh - base_pred, km, base_pred

# ---------------------------------------------------------
# 4. STRICT STEPWISE SELECTION LOGIC
# ---------------------------------------------------------
def run_selection_on_subset(df_subset, subset_name):

    df_robust = df_subset.copy()
    valid_features = [f for f in biological_features if f in df_robust.columns]
    feature_counter = Counter()

    n_cohorts = df_robust['cohort'].nunique()
    print(f"\n" + "#"*80)
    print(f"ANALYZING: {subset_name.upper()}")
    print(f"Cohorts: {n_cohorts} | Rows: {len(df_robust)}")
    print("#"*80)

    if n_cohorts < 2:
        print("SKIP: Not enough cohorts for Cross-Validation.")
        return

    # Dynamic CV splits
    n_splits = min(N_ROUNDS, n_cohorts)
    outer_cv = GroupKFold(n_splits=n_splits)

    for round_i, (train_idx, holdout_idx) in enumerate(outer_cv.split(df_robust, groups=df_robust['cohort'])):

        df_train = df_robust.iloc[train_idx].copy()
        df_holdout = df_robust.iloc[holdout_idx].copy()

        print(f"\nRound {round_i+1}/{n_splits} | Holdout: {df_holdout['cohort'].unique()}")

        # 1. Physics Baseline
        y_resid_train, train_km, base_pred_train = get_residuals(df_train)
        _, _, base_pred_holdout = get_residuals(df_holdout, km=train_km)

        # Naive Holdout Score
        naive_holdout_score = safe_spearman(df_holdout[INHIBITION], base_pred_holdout)

        # 2. Correlation Matrix
        corr_matrix = df_train[valid_features].corr().abs()

        print("-" * 110)
        print(f"{'Step':<5} | {'Added Feature':<40} | {'Train CV':<10} | {'Holdout Lift':<12} | {'Comment'}")
        print("-" * 110)

        # 3. Selection Loop
        selected_features = []
        current_pool = list(valid_features)

        # We track the "Best Train Score" to detect stagnation
        best_prev_train_score = -999.0

        for step in range(1, MAX_STEPS + 1):

            # a. Filter Candidates
            candidates = []
            for cand in current_pool:
                is_redundant = False
                for sel in selected_features:
                    if corr_matrix.loc[cand, sel] > CORR_LIMIT:
                        is_redundant = True; break
                if not is_redundant: candidates.append(cand)

            if not candidates:
                print(f"      [Stopping: No non-redundant candidates left]")
                break

            # b. Inner CV Tournament
            best_feat = None
            best_cv_score = -999.0

            # Adjust inner folds based on training set size
            n_inner = min(3, df_train['cohort'].nunique())
            # If we only have 1 training cohort, we can't do GroupKFold, so we fallback to shuffled k-fold
            if n_inner < 2:
                # Fallback: Just fit on whole train, predict on whole train (Risk of overfitting, but necessary)
                inner_cv = [(list(range(len(df_train))), list(range(len(df_train))))]
            else:
                inner_cv = GroupKFold(n_splits=n_inner)

            for cand in candidates:
                test_feats = selected_features + [cand]
                fold_scores = []

                for tr_i, val_i in inner_cv.split(df_train, groups=df_train['cohort']):
                    X_tr = df_train.iloc[tr_i][test_feats].fillna(0)
                    y_tr = y_resid_train[tr_i]
                    X_val = df_train.iloc[val_i][test_feats].fillna(0)

                    # Relaxed XGB params to ensure it learns *something* if signal exists
                    model = xgb.XGBRegressor(n_estimators=60, max_depth=4, learning_rate=0.05, n_jobs=-1)
                    model.fit(X_tr, y_tr)

                    pred_resid = model.predict(X_val)
                    final_pred = base_pred_train[val_i] + pred_resid

                    score = safe_spearman(df_train.iloc[val_i][INHIBITION], final_pred)
                    fold_scores.append(score)

                avg_score = np.mean(fold_scores)
                if avg_score > best_cv_score:
                    best_cv_score = avg_score
                    best_feat = cand

            # c. STRICT IMPROVEMENT CHECK
            # If the best new feature doesn't improve Train CV by at least 0.001, STOP.
            improvement = best_cv_score - best_prev_train_score

            if improvement < 0.001 and step > 1:
                print(f"      [Stopping: Best feature '{best_feat}' added only {improvement:.4f} to Train CV]")
                break

            best_prev_train_score = best_cv_score

            # d. Lock In & Evaluate
            selected_features.append(best_feat)
            current_pool.remove(best_feat)
            feature_counter[best_feat] += 1

            # Evaluation on Holdout
            X_tr_full = df_train[selected_features].fillna(0)
            final_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=4, n_jobs=-1)
            final_model.fit(X_tr_full, y_resid_train)

            X_hold = df_holdout[selected_features].fillna(0)
            final_pred_hold = base_pred_holdout + final_model.predict(X_hold)

            hold_score = safe_spearman(df_holdout[INHIBITION], final_pred_hold)
            lift = hold_score - naive_holdout_score

            print(f"{step:<5} | {best_feat:<40} | {best_cv_score:.4f}     | {lift:+.4f}       |")

    print(f"\nTop Features for {subset_name.upper()}:")
    for f, c in feature_counter.most_common(10):
        print(f"{f:<40} : {c}/{n_splits}")

# ---------------------------------------------------------
# 5. MAIN EXECUTION
# ---------------------------------------------------------
def run_split_analysis(merged_df):
    df_clean = merged_df.copy()
    df_clean = df_clean[df_clean[VOLUME_COL] > 0].copy()

    # Define Cohorts
    df_clean['cohort'] = df_clean[CELL_LINE].astype(str) + '_' + df_clean[CANONICAL_GENE].astype(str)

    # Filter Small
    counts = df_clean['cohort'].value_counts()
    valid_cohorts = counts[counts >= MIN_SAMPLES].index
    df_clean = df_clean[df_clean['cohort'].isin(valid_cohorts)].copy()

    # Split LIVER vs WORLD
    is_liver = df_clean[CELL_LINE].apply(lambda x: any(l in str(x) for l in LIVER_CELL_LINES))
    df_liver = df_clean[is_liver].copy()
    df_world = df_clean[~is_liver].copy()

    # --- RUN LIVER (MOE FILTERED) ---
    # Liver cohorts are mostly MOE, so this filter is safe/redundant
    # if 'moe' in df_liver.columns:
    #     df_liver_moe = df_liver[df_liver['moe'] == 1]
    #     run_selection_on_subset(df_liver_moe, "Liver Only (MOE=1)")
    # else:
    run_selection_on_subset(df_liver, "Liver Only (All Chem)")

    # --- RUN WORLD (NO FILTER) ---
    # We DO NOT apply MOE filter here, because they likely aren't MOE
    run_selection_on_subset(df_world, "The World (Non-Liver, All Chem)")

if 'merged_df' in locals():
    run_split_analysis(merged_df)


################################################################################
ANALYZING: LIVER ONLY (ALL CHEM)
Cohorts: 5 | Rows: 7204
################################################################################

Round 1/5 | Holdout: ['HepaRG_HSD17B13']
--------------------------------------------------------------------------------------------------------------
Step  | Added Feature                            | Train CV   | Holdout Lift | Comment
--------------------------------------------------------------------------------------------------------------
1     | Modification_evenness                    | 0.2657     | -0.0004       |
2     | Sequence_hairpin_tm                      | 0.3663     | -0.1376       |
3     | off_target_single_RNASEH1_c1200          | 0.3687     | -0.1339       |
      [Stopping: Best feature 'LNA_DIFF_37_HYBR' added only 0.0000 to Train CV]

Round 2/5 | Holdout: ['HepG2_DGAT2']
-----------------------------------------------------------------------

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from scipy.stats import spearmanr

# 1. CONSTANTS
from tauso.data.consts import INHIBITION, CELL_LINE, CANONICAL_GENE
VOLUME_COL = 'ASO_volume(nM)'

# 2. EXPANDED FEATURE LIST (Thermodynamics + RNase)
# We test if these drive AFFINITY (Kd), not Efficiency (Emax)
FEATURE_CANDIDATES = [
    # --- The Likely Winners (Thermodynamics) ---
    'LNA_DIFF_37_HYBR',
    'Sequence_hairpin_score',
    'access_120flank_20access_13seed_sizes',
    'CET_DIFF_37_HYBR',
    'PSDNA_RNA_MD_37_GB_TOTAL_HYBR',

    # --- The Previous Losers (RNase H1) ---
    # We re-test them in the Denominator (Maybe they affect Km?)
    'RNaseH1_score_dinucleotide_R4a_dinuc_dynamic',
    'RNaseH1_score_dinucleotide_R4b_dinuc_dynamic',
    'RNaseH1_Krel_dinucleotide_score_R4a_krel_dinuc_dynamic'
]

# 3. MODEL: Kd Predictor (Denominator)
class AffinityModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Predict Kd (Affinity) from the feature
        self.kd_net = nn.Sequential(
            nn.Linear(1, 16), nn.ReLU(),
            nn.Linear(16, 16), nn.ReLU(),
            nn.Linear(16, 1), nn.Softplus() # Kd must be positive
        )
        # Learn a global scalar for Emax (Plateau)
        # We assume plateau is constant, but let the model tune it slightly (e.g. 0.95)
        self.global_emax = nn.Parameter(torch.tensor(0.0)) # Sigmoid(0) = 0.5 start

    def forward(self, x, v):
        # Predict Kd for this specific sequence
        kd_pred = self.kd_net(x)

        # Global Emax
        emax = torch.sigmoid(self.global_emax)

        # Physics: E = Emax * V / (V + Kd)
        return emax * (v / (v + kd_pred + 1e-6))

# 4. TOURNAMENT LOOP
def run_affinity_tournament(merged_df, n_splits=3):

    results = []
    print(f"Starting AFFINITY (Denominator) Tournament...")
    print(f"{'Feature Name':<50} | {'Mean Lift':<10} | {'Std':<10}")
    print("-" * 75)

    df_clean = merged_df.copy()
    df_clean['cohort_id'] = df_clean[CELL_LINE].astype(str) + '_' + df_clean[CANONICAL_GENE].astype(str)

    for feat in FEATURE_CANDIDATES:
        if feat not in df_clean.columns: continue

        # Prep Data (One Feature)
        sub = df_clean.dropna(subset=[INHIBITION, VOLUME_COL, 'cohort_id', feat])

        # Log Transform Feature?
        # Often helpful for Energy/Score features, but let's stick to StandardScalar for now
        X = StandardScaler().fit_transform(sub[feat].values.reshape(-1, 1))

        vol = sub[VOLUME_COL].values.reshape(-1, 1)
        y = sub[INHIBITION].clip(0, 100).values.reshape(-1, 1) / 100.0
        groups = sub['cohort_id'].values

        # Cross Validation
        gss = GroupShuffleSplit(n_splits=n_splits, train_size=0.8, random_state=42)
        lifts = []

        for train_idx, test_idx in gss.split(X, y, groups):

            # Tensors
            t_x = torch.FloatTensor(X[train_idx])
            t_v = torch.FloatTensor(vol[train_idx])
            t_y = torch.FloatTensor(y[train_idx])

            test_x = torch.FloatTensor(X[test_idx])
            test_v = torch.FloatTensor(vol[test_idx])
            test_y = torch.FloatTensor(y[test_idx])

            # Baseline (Median Kd)
            tr_ec50 = t_v.numpy().flatten() * ((1.0 / t_y.numpy().flatten().clip(0.01, 0.99)) - 1.0)
            base_kd = np.median(tr_ec50)
            base_pred = test_v.numpy().flatten() / (test_v.numpy().flatten() + base_kd)
            base_score = spearmanr(test_y.numpy().flatten(), base_pred)[0]

            # Train Physics Model
            model = AffinityModel().cuda()
            opt = optim.Adam(model.parameters(), lr=0.005)
            crit = nn.MSELoss()

            loader = DataLoader(TensorDataset(t_x, t_v, t_y), batch_size=4096, shuffle=True)

            model.train()
            for _ in range(40):
                for bx, bv, by in loader:
                    opt.zero_grad()
                    loss = crit(model(bx.cuda(), bv.cuda()), by.cuda())
                    loss.backward()
                    opt.step()

            model.eval().cpu()
            with torch.no_grad():
                phys_pred = model(test_x, test_v).numpy().flatten()

            phys_score = spearmanr(test_y.numpy().flatten(), phys_pred)[0]
            lifts.append(phys_score - base_score)

        mean_lift = np.mean(lifts)
        results.append({'Feature': feat, 'Lift': mean_lift, 'Std': np.std(lifts)})
        print(f"{feat:<50} | {mean_lift:+.4f}     | {np.std(lifts):.4f}")

    # Results
    res_df = pd.DataFrame(results).sort_values('Lift', ascending=False)
    print("\n" + "="*75)
    print("WINNERS (Positive Lift = Better than Baseline)")
    print("="*75)
    print(res_df.to_string(index=False))

if 'merged_df' in locals():
    run_affinity_tournament(merged_df)

In [ ]:
# Assuming 'pipeline' is your fitted XGB_Robust pipeline from the best fold
xgb_model = pipeline.named_steps['model']
importances = xgb_model.feature_importances_
feature_names = FEATURES # Make sure this matches the X columns used

# Create a clean dataframe
imp_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("\nTop 5 Drivers of Generalization:")
print(imp_df.head(20))

In [ ]:
# Assuming 'results_df' is your detailed dataframe from the previous step
results_df['Delta'] = results_df['Model_Test_Spearman'] - results_df['Base_Test_Spearman']

print("\n--- TRUE PERFORMANCE METRICS ---")
print(f"Mean Value Added (Delta): {results_df['Delta'].mean():.4f}")
print(f"Consistency Risk (Std Dev): {results_df['Delta'].std():.4f}")

print("\n--- WHERE DID WE FAIL? (Negative Delta) ---")
failures = results_df[results_df['Delta'] < 0].sort_values('Delta')
print(failures[['Cohort', 'Base_Test_Spearman', 'Model_Test_Spearman', 'Delta']])

# Check correlation between Baseline Strength and Model Failure
corr_failure = results_df['Base_Test_Spearman'].corr(results_df['Delta'])
print(f"\nCorrelation between Baseline Strength and Value Add: {corr_failure:.4f}")
# Expect this to be strongly negative (e.g., -0.80)

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# 1. DATA PREP & LOG_EC50 DERIVATION
# ---------------------------------------------------------
df = merged_df.copy()
df['cohort'] = df[CELL_LINE].astype(str) + '_' + df[CANONICAL_GENE].astype(str)

# Using your session constants
VOLUME_COL = 'ASO_volume(nM)'
CELL_LINE_COL = 'Cell_line'
GENE_COL = CANONICAL_GENE

# Derive Potency (EC50) to strip volume bias
# Formula: EC50 = Vol * ( (1/Inh) - 1 )
inh_clipped = df[INHIBITION].clip(1, 99) / 100.0
df['log_ec50'] = np.log10(df[VOLUME_COL] * ((1.0 / inh_clipped) - 1.0))

# Standardize cohort ID
df['cohort'] = df[CELL_LINE_COL].astype(str) + '_' + df[GENE_COL].astype(str)

test_cohort_id = 'HepaRG_HSD17B13'
if test_cohort_id not in df['cohort'].unique():
    print(f"CRITICAL: {test_cohort_id} not found. Check if CELL_LINE or CANONICAL_GENE values changed.")
else:
    # Split BEFORE robust filtering to preserve test set
    df_test = df[df['cohort'] == test_cohort_id].copy()
    df_others = df[df['cohort'] != test_cohort_id].copy()

    # Apply N>=40 filter only to training data
    counts = df_others['cohort'].value_counts()
    df_dev = df_others[df_others['cohort'].isin(counts[counts >= 40].index)].copy()

    print(f"Training on {df_dev['cohort'].nunique()} robust cohorts. Testing on {test_cohort_id} (N={len(df_test)})")

    # ---------------------------------------------------------
    # 2. MECHANISTIC FEATURE ENGINEERING
    # ---------------------------------------------------------
    def engineer_physics(data, train_ref):
        # a. Unified Hybridization (Mutually exclusive LNA/cEt)
        data['Unified_Hybr'] = data['LNA_DIFF_37_HYBR'].fillna(0) + data['CET_DIFF_37_HYBR'].fillna(0)

        # b. Interaction Features (The "Biology Logic")
        # Target Accessibility * Binding Affinity
        data['Inter_Fold_Hybr'] = data['mfe_win65_flank120_step15'] * data['Unified_Hybr']
        # Affinity * mRNA Turnover
        data['Inter_Hybr_HalfLife'] = data['Unified_Hybr'] * data['mRNA_HalfLife']

        # c. Radicalized Off-Targets (Sponge Switch)
        # ReLU-style high-pass filter at the 98th percentile
        for ot in ['off_target_single_RPL7_c0', 'off_target_single_RNASEH1_c0']:
            thresh = train_ref[ot].quantile(0.98)
            data[f'Radical_{ot}'] = np.where(data[ot] > thresh, data[ot], 0.0)
        return data

    df_dev = engineer_physics(df_dev, df_dev)
    df_test = engineer_physics(df_test, df_dev)

    physics_features = [
        'Unified_Hybr',
        'mfe_win65_flank120_step15',
        'mRNA_HalfLife',
        'Inter_Fold_Hybr',
        'Inter_Hybr_HalfLife',
        'Radical_off_target_single_RPL7_c0',
        'Radical_off_target_single_RNASEH1_c0'
    ]

    # ---------------------------------------------------------
    # 3. TRAINING & EVALUATION
    # ---------------------------------------------------------
    scaler = StandardScaler()
    X_train = scaler.fit_transform(df_dev[physics_features].fillna(0))
    X_test = scaler.transform(df_test[physics_features].fillna(0))

    y_train = df_dev['log_ec50']
    y_test = df_test['log_ec50']

    # Ridge Regression for stable coefficients
    model = Ridge(alpha=10.0)
    model.fit(X_train, y_train)

    # Predict log_ec50 (Pure Potency)
    pred_log_ec50 = model.predict(X_test)
    bio_score, _ = spearmanr(y_test, pred_log_ec50)

    # Reconstruct Inhibition % (The Clinical Metric)
    pred_ec50 = 10**pred_log_ec50
    clin_preds = (df_test[VOLUME_COL] / (df_test[VOLUME_COL] + pred_ec50)) * 100
    clin_score, _ = spearmanr(df_test[INHIBITION], clin_preds)

    print("="*50)
    print(f"FORCED PHYSICS MODEL: {test_cohort_id}")
    print("-" * 50)
    print(f"Biological Spearman (EC50 Space): {bio_score:.4f}")
    print(f"Clinical Spearman   (Inh % Space): {clin_score:.4f}")
    print("="*50)

    # 4. COEFFICIENT INTERPRETATION
    # ---------------------------------------------------------
    weights = pd.Series(model.coef_, index=physics_features).sort_values()
    plt.figure(figsize=(10, 6))
    sns.barplot(x=weights.values, y=weights.index, palette='RdBu_r')
    plt.axvline(0, color='black', lw=1)
    plt.title("Mechanism Impact on log_ec50 (Negative = More Potent)")
    plt.xlabel("Standardized Coefficient Weight")
    plt.show()

In [ ]:
from tauso.data.consts import TREATMENT_PERIOD
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# 1. SETUP & TARGET DERIVATION
# ---------------------------------------------------------
df = merged_df.copy()

# Using your constants
VOLUME_COL = 'ASO_volume(nM)'
CELL_LINE_COL = 'Cell_line'
GENE_COL = CANONICAL_GENE

# Pure Potency Target (EC50)
inh_clipped = df[INHIBITION].clip(1, 99) / 100.0
df['log_ec50'] = np.log10(df[VOLUME_COL] * ((1.0 / inh_clipped) - 1.0))

# Cohort Definition
df['cohort'] = df[CELL_LINE_COL].astype(str) + '_' + df[GENE_COL].astype(str)

# Filter for robust cohorts (N >= 40)
counts = df['cohort'].value_counts()
df_robust = df[df['cohort'].isin(counts[counts >= 40].index)].copy()

# ---------------------------------------------------------
# 2. FEATURE ENGINEERING (Inside CV Loop)
# ---------------------------------------------------------
def get_physics_features(data, train_ref):
    """
    Revised Mechanistic Physics:
    - Normalizes Affinity by Length (Affinity Density)
    - Chemistry-Gated Physics (MOE vs cEt)
    - Quadratic 'Goldilocks' terms
    - Radicalized Sponge Switches
    """
    # 0. Define Length (Using the sense_length feature we saw earlier)
    # We use a floor of 1 to avoid any potential division by zero
    aso_len = data['sense_length_x'].clip(lower=1)

    # 1. Length-Normalized Hybridization (Affinity Density)
    # Dividing Delta G by N-length isolates the "Quality" of the match
    data['MOE_Hybr_Density'] = data['MOE_DIFF_37_MD_PB_HYBR'].fillna(0) / aso_len
    data['cEt_Hybr_Density'] = data['CET_DIFF_37_HYBR'].fillna(0) / aso_len

    # 2. Molecular Dynamics Non-Linearity (The Hump)
    # Using the normalized version to catch the 'Goldilocks' quality density
    data['MOE_Hybr_Density_Sq'] = data['MOE_Hybr_Density'] ** 2

    # 3. Accessibility (The KARPAS-specific bottleneck)
    acc_col = 'mfe_win45_flank120_step7'

    # 4. Mechanistic Synergy (Docking Logic)
    # Normalized binding * Accessibility
    data['Docking_MOE'] = data[acc_col] * data['MOE_Hybr_Density']
    data['Docking_cEt'] = data[acc_col] * data['cEt_Hybr_Density']

    # 5. Temporal & Biological Context
    # Clip treatment time to 48h to prevent 72h recovery noise
    data['Time_Clipped'] = np.minimum(data[TREATMENT_PERIOD], 48)
    # STANDALONE LENGTH: Does length itself (independent of energy) affect uptake/potency?
    data['ASO_Length'] = aso_len

    # 6. Radicalized Off-Targets (Catastrophic Sponge Avoidance)
    # We use the 98th percentile of the training reference to define 'Catastrophe'
    for ot in ['off_target_score_specific_MECH_n100_c1000', 'off_target_single_RPL7_c0']:
        if ot in data.columns:
            thresh = train_ref[ot].quantile(0.98)
            # High-pass filter: keep the raw value only if it is extreme, else 0
            data[f'Radical_{ot}'] = np.where(data[ot] > thresh, data[ot], 0.0)
        else:
            data[f'Radical_{ot}'] = 0.0

    # Define the final feature set for the Ridge model
    # We include mRNA_HalfLife as it is a critical gene-level offset
    return data[[
        'MOE_Hybr_Density',
        'cEt_Hybr_Density',
        'MOE_Hybr_Density_Sq',
        'ASO_Length',
        acc_col,
        'Docking_MOE',
        'Docking_cEt',
        'Time_Clipped',
        'mRNA_HalfLife',
        'Radical_off_target_single_RPL7_c0'
    ]]

# ---------------------------------------------------------
# 3. LOCO CROSS-VALIDATION LOOP
# ---------------------------------------------------------
logo = LeaveOneGroupOut()
cv_results = []

print(f"Starting LOCO CV across {df_robust['cohort'].nunique()} cohorts...")

for train_idx, val_idx in logo.split(df_robust, groups=df_robust['cohort']):
    # Split
    df_train_raw = df_robust.iloc[train_idx].copy()
    df_val_raw = df_robust.iloc[val_idx].copy()
    cohort_name = df_val_raw['cohort'].iloc[0]

    # Engineer features based on training distribution
    X_train_raw = get_physics_features(df_train_raw, df_train_raw)
    X_val_raw = get_physics_features(df_val_raw, df_train_raw)

    # Scaling
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw.fillna(0))
    X_val = scaler.transform(X_val_raw.fillna(0))

    y_train = df_train_raw['log_ec50']
    y_val = df_val_raw['log_ec50']

    # Fit Ridge
    model = Ridge(alpha=10.0)
    model.fit(X_train, y_train)

    # Predict Potency
    pred_log_ec50 = model.predict(X_val)

    # Scores
    bio_corr, _ = spearmanr(y_val, pred_log_ec50)

    # Reconstruct Clinical Score (Inh %)
    pred_ec50 = 10**pred_log_ec50
    clin_preds = (df_val_raw[VOLUME_COL] / (df_val_raw[VOLUME_COL] + pred_ec50)) * 100
    clin_corr, _ = spearmanr(df_val_raw[INHIBITION], clin_preds)

    cv_results.append({
        'cohort': cohort_name,
        'bio_spearman': bio_corr,
        'clin_spearman': clin_corr,
        'N': len(df_val_raw)
    })

# ---------------------------------------------------------
# 4. VISUALIZATION OF GENERALIZATION
# ---------------------------------------------------------
results_df = pd.DataFrame(cv_results).sort_values('bio_spearman', ascending=False)

plt.figure(figsize=(12, 7))
sns.barplot(data=results_df, x='bio_spearman', y='cohort', palette='viridis')
plt.axvline(0, color='black', lw=1)
plt.axvline(results_df['bio_spearman'].mean(), color='red', linestyle='--',
            label=f"Mean Bio Spearman: {results_df['bio_spearman'].mean():.3f}")

plt.title("Cross-Validation: Ability to Predict Potency (EC50) on Unseen Cohorts")
plt.xlabel("Spearman Correlation (Predicted vs Actual log_ec50)")
plt.legend()
plt.tight_layout()
plt.show()

print("\nSummary Statistics:")
print(results_df[['bio_spearman', 'clin_spearman']].describe())